In [7]:
import os
os.listdir('/home/jovyan/work/jars')

['clickhouse-jdbc-0.6.3.jar', 'postgresql-42.7.3.jar']

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab2 Star to ClickHouse") \
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/postgresql-42.7.3.jar,/home/jovyan/work/jars/clickhouse-jdbc-0.6.3.jar"
    ) \
    .getOrCreate()

In [9]:
pg_url = "jdbc:postgresql://postgres:5432/lab2_db"

pg_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

In [10]:
ch_url = "jdbc:clickhouse://clickhouse:8123/default?compress=0"

ch_properties = {
    "user": "default",
    "password": "clickhouse",
    "driver": "com.clickhouse.jdbc.ClickHouseDriver"
}

In [11]:
test_ch_df = spark.createDataFrame(
    [(1, "ok")],
    ["id", "status"]
)

In [12]:
test_ch_df.write.jdbc(
    url=ch_url,
    table="test_clickhouse_connection",
    mode="overwrite",
    properties=ch_properties
)

In [13]:
dim_customer_df = spark.read.jdbc(pg_url, "dim_customer", properties=pg_properties)
dim_seller_df = spark.read.jdbc(pg_url, "dim_seller", properties=pg_properties)
dim_store_df = spark.read.jdbc(pg_url, "dim_store", properties=pg_properties)
dim_supplier_df = spark.read.jdbc(pg_url, "dim_supplier", properties=pg_properties)
dim_product_df = spark.read.jdbc(pg_url, "dim_product", properties=pg_properties)
dim_date_df = spark.read.jdbc(pg_url, "dim_date", properties=pg_properties)
fact_sales_df = spark.read.jdbc(pg_url, "fact_sales", properties=pg_properties)

In [14]:
print("dim_customer:", dim_customer_df.count())
print("dim_seller:", dim_seller_df.count())
print("dim_store:", dim_store_df.count())
print("dim_supplier:", dim_supplier_df.count())
print("dim_product:", dim_product_df.count())
print("dim_date:", dim_date_df.count())
print("fact_sales:", fact_sales_df.count())

dim_customer: 10000
dim_seller: 10000
dim_store: 10000
dim_supplier: 10000
dim_product: 9742
dim_date: 364
fact_sales: 10000


In [30]:
sales_full_df = (
    fact_sales_df.alias("f")
    .join(dim_customer_df.alias("c"), "customer_id", "left")
    .join(dim_seller_df.alias("s"), "seller_id", "left")
    .join(dim_store_df.alias("st"), "store_id", "left")
    .join(dim_supplier_df.alias("sp"), "supplier_id", "left")
    .join(dim_product_df.alias("p"), "product_id", "left")
    .join(dim_date_df.alias("d"), "date_id", "left")
    .select(
        col("f.date_id"),
        col("f.customer_id"),
        col("f.seller_id"),
        col("f.product_id"),
        col("f.store_id"),
        col("f.supplier_id"),
        col("f.sale_quantity"),
        col("f.sale_total_price"),

        col("c.first_name").alias("customer_first_name"),
        col("c.last_name").alias("customer_last_name"),
        col("c.email").alias("customer_email"),
        col("c.country").alias("customer_country"),
        col("c.postal_code").alias("customer_postal_code"),

        col("s.first_name").alias("seller_first_name"),
        col("s.last_name").alias("seller_last_name"),
        col("s.email").alias("seller_email"),
        col("s.country").alias("seller_country"),
        col("s.postal_code").alias("seller_postal_code"),

        col("st.store_name"),
        col("st.store_location"),
        col("st.store_city"),
        col("st.store_state"),
        col("st.store_country"),
        col("st.store_phone"),
        col("st.store_email"),

        col("sp.supplier_name"),
        col("sp.supplier_contact"),
        col("sp.supplier_email"),
        col("sp.supplier_phone"),
        col("sp.supplier_address"),
        col("sp.supplier_city"),
        col("sp.supplier_country"),

        col("p.product_name"),
        col("p.product_category"),
        col("p.product_brand"),
        col("p.product_price"),
        col("p.product_quantity"),
        col("p.product_weight"),
        col("p.product_color"),
        col("p.product_size"),
        col("p.product_material"),
        col("p.product_description"),
        col("p.product_rating"),
        col("p.product_reviews"),
        col("p.product_release_date"),
        col("p.product_expiry_date"),
        col("p.pet_category"),

        col("d.full_date"),
        col("d.day"),
        col("d.month"),
        col("d.year"),
        col("d.quarter")
    )
)

In [31]:
sales_full_df.printSchema()

root
 |-- date_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- seller_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- sale_quantity: integer (nullable = true)
 |-- sale_total_price: decimal(10,2) (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- store_state: string (

In [32]:
sales_full_df.show(5, truncate=False)

+-------+-----------+---------+----------+--------+-----------+-------------+----------------+-------------------+------------------+-----------------------------+----------------+--------------------+-----------------+----------------+------------------------+--------------+------------------+----------+--------------+----------+-----------+-------------+------------+----------------------+-------------+----------------+------------------------+--------------+----------------+-------------+----------------+------------+----------------+-------------+-------------+----------------+--------------+-------------+------------+----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [33]:
sales_full_df.count()

10000

In [34]:
from pyspark.sql.functions import sum, max, avg, col, count, dense_rank
from pyspark.sql.window import Window

In [35]:
product_sales_df = (
    sales_full_df
    .groupBy(
        "product_id",
        "product_name",
        "product_category",
        "product_brand"
    )
    .agg(
        sum("sale_quantity").alias("total_sales_qty"),
        sum("sale_total_price").alias("total_revenue"),
        count("*").alias("sales_count"),
        avg("product_rating").alias("avg_rating"),
        max("product_reviews").alias("total_reviews")
    )
)

In [36]:
category_revenue_df = (
    sales_full_df
    .groupBy("product_category")
    .agg(
        sum("sale_total_price").alias("category_revenue")
    )
)

In [37]:
product_sales_report_df = (
    product_sales_df
    .join(category_revenue_df, on="product_category", how="left")
)

In [38]:
product_rank_window = Window.orderBy(col("total_sales_qty").desc())

product_sales_report_df = (
    product_sales_report_df
    .withColumn("product_rank_by_sales", dense_rank().over(product_rank_window))
)

In [39]:
product_sales_report_df.printSchema()

root
 |-- product_category: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- total_sales_qty: long (nullable = true)
 |-- total_revenue: decimal(20,2) (nullable = true)
 |-- sales_count: long (nullable = false)
 |-- avg_rating: decimal(7,5) (nullable = true)
 |-- total_reviews: integer (nullable = true)
 |-- category_revenue: decimal(20,2) (nullable = true)
 |-- product_rank_by_sales: integer (nullable = false)



In [40]:
product_sales_report_df.show(10, truncate=False)

+----------------+----------+------------+-------------+---------------+-------------+-----------+----------+-------------+----------------+---------------------+
|product_category|product_id|product_name|product_brand|total_sales_qty|total_revenue|sales_count|avg_rating|total_reviews|category_revenue|product_rank_by_sales|
+----------------+----------+------------+-------------+---------------+-------------+-----------+----------+-------------+----------------+---------------------+
|Cage            |1031      |Bird Cage   |Yodel        |21             |554.10       |3          |2.40000   |32           |831117.94       |1                    |
|Toy             |2173      |Bird Cage   |Aivee        |20             |765.84       |2          |4.80000   |966          |868101.63       |2                    |
|Food            |7695      |Dog Food    |Dabfeed      |20             |451.13       |2          |1.70000   |437          |830632.55       |2                    |
|Toy             |6414

In [41]:
product_sales_report_df.orderBy(col("total_sales_qty").desc()).show(10, truncate=False)

+----------------+----------+------------+-------------+---------------+-------------+-----------+----------+-------------+----------------+---------------------+
|product_category|product_id|product_name|product_brand|total_sales_qty|total_revenue|sales_count|avg_rating|total_reviews|category_revenue|product_rank_by_sales|
+----------------+----------+------------+-------------+---------------+-------------+-----------+----------+-------------+----------------+---------------------+
|Cage            |1031      |Bird Cage   |Yodel        |21             |554.10       |3          |2.40000   |32           |831117.94       |1                    |
|Toy             |2173      |Bird Cage   |Aivee        |20             |765.84       |2          |4.80000   |966          |868101.63       |2                    |
|Food            |7695      |Dog Food    |Dabfeed      |20             |451.13       |2          |1.70000   |437          |830632.55       |2                    |
|Toy             |6414

In [42]:
import os
os.listdir('/home/jovyan/work/jars')

['clickhouse-jdbc-0.6.3.jar', 'postgresql-42.7.3.jar']

In [43]:
product_sales_report_df.write.jdbc(
    url=ch_url,
    table="report_product_sales",
    mode="overwrite",
    properties=ch_properties
)

In [45]:
customer_sales_df = (
    sales_full_df
    .groupBy(
        "customer_id",
        "customer_first_name",
        "customer_last_name",
        "customer_email",
        "customer_country"
    )
    .agg(
        sum("sale_total_price").alias("total_spent"),
        count("*").alias("orders_count"),
        avg("sale_total_price").alias("avg_check")
    )
)

In [46]:
country_customer_df = (
    dim_customer_df
    .groupBy(col("country").alias("customer_country"))
    .agg(count("*").alias("country_customer_count"))
)

In [47]:
customer_sales_report_df = (
    customer_sales_df
    .join(country_customer_df, on="customer_country", how="left")
)

In [48]:
customer_rank_window = Window.orderBy(col("total_spent").desc())

customer_sales_report_df = (
    customer_sales_report_df
    .withColumn("customer_rank_by_spent", dense_rank().over(customer_rank_window))
)

In [49]:
customer_sales_report_df.show(10, truncate=False)

+----------------+-----------+-------------------+------------------+----------------------------+-----------+------------+----------+----------------------+----------------------+
|customer_country|customer_id|customer_first_name|customer_last_name|customer_email              |total_spent|orders_count|avg_check |country_customer_count|customer_rank_by_spent|
+----------------+-----------+-------------------+------------------+----------------------------+-----------+------------+----------+----------------------+----------------------+
|Albania         |1025       |Gus                |Hartshorn         |bfeasby57@youku.com         |499.85     |1           |499.850000|46                    |1                     |
|Portugal        |8868       |Hayes              |McKain            |sstappardbp@businessweek.com|499.80     |1           |499.800000|336                   |2                     |
|China           |2851       |Ava                |Lomas             |dsorea0@geocities.com     

In [50]:
customer_sales_report_df.write.jdbc(
    url=ch_url,
    table="report_customer_sales",
    mode="overwrite",
    properties=ch_properties
)

In [51]:
time_sales_report_df = (
    sales_full_df
    .groupBy("year", "month", "quarter")
    .agg(
        count("*").alias("orders_count"),
        sum("sale_quantity").alias("total_sales_qty"),
        sum("sale_total_price").alias("total_revenue"),
        avg("sale_quantity").alias("avg_order_size"),
        avg("sale_total_price").alias("avg_order_value")
    )
    .orderBy("year", "month")
)

In [52]:
time_sales_report_df.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- orders_count: long (nullable = false)
 |-- total_sales_qty: long (nullable = true)
 |-- total_revenue: decimal(20,2) (nullable = true)
 |-- avg_order_size: double (nullable = true)
 |-- avg_order_value: decimal(14,6) (nullable = true)



In [53]:
time_sales_report_df.show(20, truncate=False)

+----+-----+-------+------------+---------------+-------------+------------------+---------------+
|year|month|quarter|orders_count|total_sales_qty|total_revenue|avg_order_size    |avg_order_value|
+----+-----+-------+------------+---------------+-------------+------------------+---------------+
|2021|1    |1      |874         |4856           |224158.54    |5.556064073226545 |256.474302     |
|2021|2    |1      |739         |4070           |192348.31    |5.50744248985115  |260.281881     |
|2021|3    |1      |843         |4561           |207282.20    |5.4104389086595495|245.886358     |
|2021|4    |2      |837         |4564           |206592.82    |5.4528076463560335|246.825352     |
|2021|5    |2      |828         |4451           |211764.86    |5.375603864734299 |255.754662     |
|2021|6    |2      |822         |4438           |215042.80    |5.399026763990268 |261.609246     |
|2021|7    |3      |858         |4750           |220496.51    |5.536130536130536 |256.988939     |
|2021|8   

In [54]:
time_sales_report_df.write.jdbc(
    url=ch_url,
    table="report_time_sales",
    mode="overwrite",
    properties=ch_properties
)

In [55]:
store_sales_df = (
    sales_full_df
    .groupBy(
        "store_id",
        "store_name",
        "store_city",
        "store_country"
    )
    .agg(
        sum("sale_total_price").alias("total_revenue"),
        count("*").alias("orders_count"),
        sum("sale_quantity").alias("total_sales_qty"),
        avg("sale_total_price").alias("avg_check")
    )
)

In [56]:
city_revenue_df = (
    sales_full_df
    .groupBy("store_city")
    .agg(
        sum("sale_total_price").alias("city_revenue")
    )
)

In [57]:
country_revenue_df = (
    sales_full_df
    .groupBy("store_country")
    .agg(
        sum("sale_total_price").alias("country_revenue")
    )
)

In [58]:
store_sales_report_df = (
    store_sales_df
    .join(city_revenue_df, on="store_city", how="left")
    .join(country_revenue_df, on="store_country", how="left")
)

In [59]:
store_rank_window = Window.orderBy(col("total_revenue").desc())

store_sales_report_df = (
    store_sales_report_df
    .withColumn("store_rank_by_revenue", dense_rank().over(store_rank_window))
)

In [60]:
store_sales_report_df.printSchema()

root
 |-- store_country: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- total_revenue: decimal(20,2) (nullable = true)
 |-- orders_count: long (nullable = false)
 |-- total_sales_qty: long (nullable = true)
 |-- avg_check: decimal(14,6) (nullable = true)
 |-- city_revenue: decimal(20,2) (nullable = true)
 |-- country_revenue: decimal(20,2) (nullable = true)
 |-- store_rank_by_revenue: integer (nullable = false)



In [61]:
store_sales_report_df.show(10, truncate=False)

+-------------+----------+--------+-----------+-------------+------------+---------------+----------+------------+---------------+---------------------+
|store_country|store_city|store_id|store_name |total_revenue|orders_count|total_sales_qty|avg_check |city_revenue|country_revenue|store_rank_by_revenue|
+-------------+----------+--------+-----------+-------------+------------+---------------+----------+------------+---------------+---------------------+
|South Africa |Grekan    |1025    |DabZ       |499.85       |1           |7              |499.850000|739.23      |20072.29       |1                    |
|Poland       |Fonte     |8868    |Thoughtblab|499.80       |1           |9              |499.800000|499.80      |81313.13       |2                    |
|Sweden       |Longzhong |2851    |Camido     |499.76       |1           |2              |499.760000|805.93      |64148.78       |3                    |
|Indonesia    |Pesek     |8048    |Edgeblab   |499.76       |1           |8       

In [62]:
store_sales_report_df.orderBy(col("total_revenue").desc()).show(5, truncate=False)

+-------------+----------+--------+-----------+-------------+------------+---------------+----------+------------+---------------+---------------------+
|store_country|store_city|store_id|store_name |total_revenue|orders_count|total_sales_qty|avg_check |city_revenue|country_revenue|store_rank_by_revenue|
+-------------+----------+--------+-----------+-------------+------------+---------------+----------+------------+---------------+---------------------+
|South Africa |Grekan    |1025    |DabZ       |499.85       |1           |7              |499.850000|739.23      |20072.29       |1                    |
|Poland       |Fonte     |8868    |Thoughtblab|499.80       |1           |9              |499.800000|499.80      |81313.13       |2                    |
|Sweden       |Longzhong |2851    |Camido     |499.76       |1           |2              |499.760000|805.93      |64148.78       |3                    |
|Indonesia    |Pesek     |8048    |Edgeblab   |499.76       |1           |8       

In [63]:
store_sales_report_df.write.jdbc(
    url=ch_url,
    table="report_store_sales",
    mode="overwrite",
    properties=ch_properties
)

In [64]:
supplier_sales_df = (
    sales_full_df
    .groupBy(
        "supplier_id",
        "supplier_name",
        "supplier_country"
    )
    .agg(
        sum("sale_total_price").alias("total_revenue"),
        count("*").alias("orders_count"),
        sum("sale_quantity").alias("total_sales_qty"),
        avg("product_price").alias("avg_product_price")
    )
)

In [65]:
supplier_country_revenue_df = (
    sales_full_df
    .groupBy("supplier_country")
    .agg(
        sum("sale_total_price").alias("country_revenue")
    )
)

In [66]:
supplier_sales_report_df = (
    supplier_sales_df
    .join(supplier_country_revenue_df, on="supplier_country", how="left")
)

In [67]:
supplier_rank_window = Window.orderBy(col("total_revenue").desc())

supplier_sales_report_df = (
    supplier_sales_report_df
    .withColumn("supplier_rank_by_revenue", dense_rank().over(supplier_rank_window))
)

In [68]:
supplier_sales_report_df.printSchema()

root
 |-- supplier_country: string (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- total_revenue: decimal(20,2) (nullable = true)
 |-- orders_count: long (nullable = false)
 |-- total_sales_qty: long (nullable = true)
 |-- avg_product_price: decimal(14,6) (nullable = true)
 |-- country_revenue: decimal(20,2) (nullable = true)
 |-- supplier_rank_by_revenue: integer (nullable = false)



In [69]:
supplier_sales_report_df.show(10, truncate=False)

+----------------+-----------+-------------+-------------+------------+---------------+-----------------+---------------+------------------------+
|supplier_country|supplier_id|supplier_name|total_revenue|orders_count|total_sales_qty|avg_product_price|country_revenue|supplier_rank_by_revenue|
+----------------+-----------+-------------+-------------+------------+---------------+-----------------+---------------+------------------------+
|Ireland         |1025       |Brainverse   |499.85       |1           |7              |31.420000        |11636.18       |1                       |
|Russia          |8868       |Jamia        |499.80       |1           |9              |6.640000         |149206.75      |2                       |
|Portugal        |2851       |Eabox        |499.76       |1           |2              |80.910000        |83210.60       |3                       |
|China           |8048       |Demimbu      |499.76       |1           |8              |53.800000        |492823.31    

In [70]:
supplier_sales_report_df.orderBy(col("total_revenue").desc()).show(5, truncate=False)

+----------------+-----------+-------------+-------------+------------+---------------+-----------------+---------------+------------------------+
|supplier_country|supplier_id|supplier_name|total_revenue|orders_count|total_sales_qty|avg_product_price|country_revenue|supplier_rank_by_revenue|
+----------------+-----------+-------------+-------------+------------+---------------+-----------------+---------------+------------------------+
|Ireland         |1025       |Brainverse   |499.85       |1           |7              |31.420000        |11636.18       |1                       |
|Russia          |8868       |Jamia        |499.80       |1           |9              |6.640000         |149206.75      |2                       |
|China           |8048       |Demimbu      |499.76       |1           |8              |53.800000        |492823.31      |3                       |
|Portugal        |2851       |Eabox        |499.76       |1           |2              |80.910000        |83210.60     

In [71]:
supplier_sales_report_df.write.jdbc(
    url=ch_url,
    table="report_supplier_sales",
    mode="overwrite",
    properties=ch_properties
)

In [72]:
product_quality_df = (
    sales_full_df
    .groupBy(
        "product_id",
        "product_name",
        "product_category",
        "product_brand",
        "product_rating",
        "product_reviews"
    )
    .agg(
        sum("sale_quantity").alias("total_sales_qty"),
        sum("sale_total_price").alias("total_revenue")
    )
)

In [73]:
sales_rating_correlation = product_quality_df.stat.corr("product_rating", "total_sales_qty")
sales_rating_correlation

-0.0005637282798695828

In [74]:
rating_desc_window = Window.orderBy(col("product_rating").desc())
rating_asc_window = Window.orderBy(col("product_rating").asc())
reviews_desc_window = Window.orderBy(col("product_reviews").desc())

product_quality_report_df = (
    product_quality_df
    .withColumn("rating_rank_desc", dense_rank().over(rating_desc_window))
    .withColumn("rating_rank_asc", dense_rank().over(rating_asc_window))
    .withColumn("reviews_rank_desc", dense_rank().over(reviews_desc_window))
)

In [75]:
from pyspark.sql.functions import lit

product_quality_report_df = (
    product_quality_report_df
    .withColumn("sales_rating_correlation", lit(sales_rating_correlation))
)

In [76]:
product_quality_report_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_rating: decimal(3,1) (nullable = true)
 |-- product_reviews: integer (nullable = true)
 |-- total_sales_qty: long (nullable = true)
 |-- total_revenue: decimal(20,2) (nullable = true)
 |-- rating_rank_desc: integer (nullable = false)
 |-- rating_rank_asc: integer (nullable = false)
 |-- reviews_rank_desc: integer (nullable = false)
 |-- sales_rating_correlation: double (nullable = false)



In [77]:
product_quality_report_df.show(10, truncate=False)

+----------+------------+----------------+-------------+--------------+---------------+---------------+-------------+----------------+---------------+-----------------+------------------------+
|product_id|product_name|product_category|product_brand|product_rating|product_reviews|total_sales_qty|total_revenue|rating_rank_desc|rating_rank_asc|reviews_rank_desc|sales_rating_correlation|
+----------+------------+----------------+-------------+--------------+---------------+---------------+-------------+----------------+---------------+-----------------+------------------------+
|4908      |Cat Toy     |Food            |Oozz         |2.1           |1000           |8              |58.71        |30              |12             |1                |-5.637282798695828E-4   |
|5640      |Cat Toy     |Toy             |Edgeify      |1.6           |1000           |4              |162.89       |35              |7              |1                |-5.637282798695828E-4   |
|2460      |Bird Cage   |Toy  

In [78]:
product_quality_report_df.orderBy(col("product_rating").desc()).show(10, truncate=False)

+----------+------------+----------------+-------------+--------------+---------------+---------------+-------------+----------------+---------------+-----------------+------------------------+
|product_id|product_name|product_category|product_brand|product_rating|product_reviews|total_sales_qty|total_revenue|rating_rank_desc|rating_rank_asc|reviews_rank_desc|sales_rating_correlation|
+----------+------------+----------------+-------------+--------------+---------------+---------------+-------------+----------------+---------------+-----------------+------------------------+
|9732      |Dog Food    |Toy             |Zoovu        |5.0           |966            |1              |410.77       |1               |41             |35               |-5.637282798695828E-4   |
|4212      |Cat Toy     |Cage            |Voomm        |5.0           |851            |10             |308.17       |1               |41             |150              |-5.637282798695828E-4   |
|489       |Bird Cage   |Cage 

In [79]:
product_quality_report_df.orderBy(col("product_rating").asc()).show(10, truncate=False)

+----------+------------+----------------+-------------+--------------+---------------+---------------+-------------+----------------+---------------+-----------------+------------------------+
|product_id|product_name|product_category|product_brand|product_rating|product_reviews|total_sales_qty|total_revenue|rating_rank_desc|rating_rank_asc|reviews_rank_desc|sales_rating_correlation|
+----------+------------+----------------+-------------+--------------+---------------+---------------+-------------+----------------+---------------+-----------------+------------------------+
|3771      |Cat Toy     |Cage            |Mycat        |1.0           |952            |7              |391.19       |41              |1              |49               |-5.637282798695828E-4   |
|5484      |Cat Toy     |Toy             |Brightbean   |1.0           |854            |4              |421.96       |41              |1              |147              |-5.637282798695828E-4   |
|1870      |Bird Cage   |Food 

In [80]:
product_quality_report_df.orderBy(col("product_reviews").desc()).show(10, truncate=False)

+----------+------------+----------------+-------------+--------------+---------------+---------------+-------------+----------------+---------------+-----------------+------------------------+
|product_id|product_name|product_category|product_brand|product_rating|product_reviews|total_sales_qty|total_revenue|rating_rank_desc|rating_rank_asc|reviews_rank_desc|sales_rating_correlation|
+----------+------------+----------------+-------------+--------------+---------------+---------------+-------------+----------------+---------------+-----------------+------------------------+
|5640      |Cat Toy     |Toy             |Edgeify      |1.6           |1000           |4              |162.89       |35              |7              |1                |-5.637282798695828E-4   |
|2460      |Bird Cage   |Toy             |Flashspan    |1.6           |1000           |7              |256.17       |35              |7              |1                |-5.637282798695828E-4   |
|7891      |Dog Food    |Food 

In [81]:
product_quality_report_df.write.jdbc(
    url=ch_url,
    table="report_product_quality",
    mode="overwrite",
    properties=ch_properties
)